# Distance to k-Nearest Pharmacies from SAL Centroids
## Euclidean and OSMnx Network Distance (Pedestrian + Drive), k = 3

**Tess Vu**

This notebook computes the distance from each Small Area Layer (SAL)
centroid to the **three nearest** geocoded pharmacies using three methods:

1. **Euclidean** (straight-line) distance via scipy KDTree.
2. **Pedestrian network** distance via OSMnx walk graph.
3. **Drive network** distance via OSMnx drive graph.

k=3 captures access redundancy and choice beyond simple reachability.
A SAL with one pharmacy at 2.9 km passes a 3 km threshold, but if that
pharmacy closes or stocks out, the community is isolated. Three nearest
pharmacies reveal whether fallback options exist.

**Compute note:** k=1 used multi-source Dijkstra (one pass from all
pharmacy nodes). k=3 requires single-source Dijkstra from each pharmacy
node individually, which is substantially slower. Checkpointing is built
in so interrupted runs can resume without losing progress.

## SETUP AND IMPORTS

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from scipy.spatial import cKDTree
from shapely.geometry import Point
from matplotlib.colors import LinearSegmentedColormap
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib as mpl
import mapclassify
import warnings
import os
import time
import pickle
import heapq

warnings.filterwarnings("ignore")
print(f"OSMnx version: {ox.__version__}")
print(f"NetworkX version: {nx.__version__}")

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import dair_style as ds
IMAGES = Path("images") / "sal_pharmacy_distance_k3"

FONT_TITLE = ds.FONT_TITLE
FONT_BODY = ds.FONT_BODY

IN_SAL_DEDUP = INTERMEDIATE / "sal_w_ward_dedup" / "sal_w_ward_dedup.shp"
IN_POP_PRED = INTERMEDIATE / "pop_pred_final.csv"
IN_PHARMACIES = RAW / "PHARMACIES_MASTER_FINAL.csv"
IN_NETWORK_DIR = INTERMEDIATE / "networks"
OUT_DISTANCES = INTERMEDIATE / "sal_pharmacy_distances_k3.csv"
OUT_INTERIM_DIR = INTERMEDIATE / "networks"
OUT_CHECKPOINT_DIR = INTERMEDIATE / "networks" / "checkpoints"


In [ ]:
# Define custom color ramp for heatmaps.
# Distance ramp from the shared DAIR style.
custom_cmap = ds.DISTANCE_CMAP

def reformat_legend_labels(ax):
    """Reformat geopandas interval legend labels:
    'a, b' -> 'a - b', and collapse '-inf, x' or 'x, x' -> just 'x'.
    """
    legend = ax.get_legend()
    if legend is None:
        return
    for text in legend.get_texts():
        label = text.get_text()
        if ", " in label:
            parts = label.split(", ", 1)
            lo, hi = parts[0].strip(), parts[1].strip()
            if lo == hi or "inf" in lo.lower():
                text.set_text(hi)
            else:
                text.set_text(f"{lo} - {hi}")


## CONFIGURATION

All file paths, distance parameters, and checkpoint settings
in a single cell. Update paths to match local directory structure.

In [ ]:
# FILE PATHS
# All input and output paths are defined in the paths cell above.
# Output directory for results and interim files.
os.makedirs(OUT_INTERIM_DIR, exist_ok = True)

# Projected CRS for distance calculations in meters.
PROJECTED_CRS = "EPSG:32735"

# Province names for filtering and labeling.
PROVINCES = ["Gauteng", "KwaZulu-Natal"]

# K-NEAREST PARAMETER
# Number of nearest pharmacies to compute for each SAL centroid.
K_NEAREST = 3

# NETWORK DISTANCE CUTOFF (meters)
# Dijkstra stops exploring beyond this distance from each pharmacy node.
# 50 km is generous; keeps compute bounded without missing realistic access.
CUTOFF_M = 50000

# CHECKPOINT SETTINGS
# Directory for intermediate checkpoint files.
os.makedirs(OUT_CHECKPOINT_DIR, exist_ok = True)

# Save checkpoint after every N pharmacy nodes processed.
CHECKPOINT_INTERVAL = 50

# Expected graphml file paths (must already exist on disk).
GRAPHML_PATHS = {
    "Gauteng_walk": IN_NETWORK_DIR / "network_gauteng_walk.graphml",
    "Gauteng_drive": IN_NETWORK_DIR / "network_gauteng_drive.graphml",
    "KwaZulu-Natal_walk": IN_NETWORK_DIR / "network_kwazulu_natal_walk.graphml",
    "KwaZulu-Natal_drive": IN_NETWORK_DIR / "network_kwazulu_natal_drive.graphml",
}

print(f"K_NEAREST: {K_NEAREST}")
print(f"CUTOFF: {CUTOFF_M / 1000:.0f} km")
print(f"CHECKPOINT INTERVAL: every {CHECKPOINT_INTERVAL} pharmacies")
print(f"OUTPUT DIR: {os.path.abspath(OUT_INTERIM_DIR)}")


## LOAD SAL GEOMETRIES AND POPULATION ESTIMATES

Load the SAL shapefile that has ward assignments, then join
with the population estimates produced by the dasymetric notebook.

In [ ]:
# Load SAL shapefile with geometry and ward assignments.
sal_geo = gpd.read_file(IN_SAL_DEDUP)
print(f"SAL SHAPEFILE LOADED: {sal_geo.shape[0]} features")
print(f"CRS: {sal_geo.crs}")
print(f"Columns: {list(sal_geo.columns)}")

In [ ]:
# Load population estimates from the dasymetric step-down.
pop_est = pd.read_csv(IN_POP_PRED)
print(f"POPULATION ESTIMATES LOADED: {pop_est.shape[0]} rows")
print(f"Columns: {list(pop_est.columns)}")

In [ ]:
# Join population estimates onto SAL geometries.
# The join key is EA_CODE (SAL identifier).
sal = sal_geo.merge(
    pop_est[["EA_CODE", "sal2023_est", "WardID"]],
    on = "EA_CODE",
    how = "left"
)

# Report join quality.
matched = sal["sal2023_est"].notna().sum()
total = sal.shape[0]
print(f"JOIN RESULTS: {matched}/{total} SALs matched ({matched / total * 100:.1f}%)")

# Drop SALs with no population estimate.
sal = sal[sal["sal2023_est"].notna()].copy()
print(f"SALs RETAINED AFTER DROPPING UNMATCHED: {sal.shape[0]}")

## COMPUTE SAL CENTROIDS

Project to a metric CRS, compute geometric centroids, then extract
coordinates for distance calculations.

**Limitation:** Geometric centroids may fall in uninhabited areas
(rivers, parks, industrial zones) for irregularly shaped SALs.
Population-weighted centroids using building footprints would be
more defensible but require additional data processing.

In [ ]:
# Project SAL geometries to metric CRS for distance calculations.
sal_proj = sal.to_crs(PROJECTED_CRS)

# Compute geometric centroids in projected space.
sal_proj["centroid_geom"] = sal_proj.geometry.centroid

# Extract centroid coordinates in projected CRS (meters).
sal_proj["centroid_x"] = sal_proj["centroid_geom"].x
sal_proj["centroid_y"] = sal_proj["centroid_geom"].y

# Also compute centroids in WGS84 for OSMnx (which expects lat/lng).
sal_wgs84 = sal.to_crs("EPSG:4326")
sal_wgs84["centroid_wgs84"] = sal_wgs84.geometry.centroid
sal_proj["centroid_lat"] = sal_wgs84["centroid_wgs84"].y.values
sal_proj["centroid_lng"] = sal_wgs84["centroid_wgs84"].x.values

print(f"SAL CENTROIDS COMPUTED: {sal_proj.shape[0]} centroids")
print(f"Projected CRS: {PROJECTED_CRS}")
print(f"Centroid X range: {sal_proj['centroid_x'].min():.0f} to {sal_proj['centroid_x'].max():.0f} m")
print(f"Centroid Y range: {sal_proj['centroid_y'].min():.0f} to {sal_proj['centroid_y'].max():.0f} m")

## LOAD GEOCODED PHARMACIES

Load the fully geocoded pharmacy file and convert to a GeoDataFrame.

In [ ]:
# Load geocoded pharmacies.
pharm_df = pd.read_csv(IN_PHARMACIES)
print(f"PHARMACIES LOADED: {pharm_df.shape[0]} rows")
print(f"Columns: {list(pharm_df.columns)}")

# Check for lat/lng columns and filter valid coordinates.
assert "LAT" in pharm_df.columns, "Missing 'LAT' column in pharmacy file."
assert "LNG" in pharm_df.columns, "Missing 'LNG' column in pharmacy file."

# Drop rows with missing coordinates.
valid_coords = pharm_df["LAT"].notna() & pharm_df["LNG"].notna()
print(f"Pharmacies with valid coordinates: {valid_coords.sum()}/{pharm_df.shape[0]}")
pharm_df = pharm_df[valid_coords].copy()

# Convert to GeoDataFrame.
pharm_gdf = gpd.GeoDataFrame(
    pharm_df,
    geometry = gpd.points_from_xy(pharm_df["LNG"], pharm_df["LAT"]),
    crs = "EPSG:4326"
)

# Project to metric CRS.
pharm_proj = pharm_gdf.to_crs(PROJECTED_CRS)
pharm_proj["pharm_x"] = pharm_proj.geometry.x
pharm_proj["pharm_y"] = pharm_proj.geometry.y

print(f"PHARMACIES READY: {pharm_proj.shape[0]} geocoded locations")

In [ ]:
# Quick sanity check: bounding box of pharmacies vs SALs.
print("BOUNDING BOX COMPARISON (WGS84)")
sal_bounds = sal.to_crs("EPSG:4326").total_bounds
pharm_bounds = pharm_gdf.total_bounds
print(f"SALs: W={sal_bounds[0]:.3f}, S={sal_bounds[1]:.3f}, E={sal_bounds[2]:.3f}, N={sal_bounds[3]:.3f}")
print(f"Pharmacies: W={pharm_bounds[0]:.3f}, S={pharm_bounds[1]:.3f}, E={pharm_bounds[2]:.3f}, N={pharm_bounds[3]:.3f}")

## EUCLIDEAN DISTANCE TO k-NEAREST PHARMACIES

Use a KDTree for efficient k-nearest-neighbor lookup in projected
coordinates (meters). This gives straight-line distances that
underestimate true travel distances but serve as a useful lower bound.
KDTree handles k > 1 natively with negligible additional cost.

In [ ]:
# Build KDTree from pharmacy locations in projected CRS.
pharm_coords = np.column_stack([pharm_proj["pharm_x"].values, pharm_proj["pharm_y"].values])
tree = cKDTree(pharm_coords)

# Query k nearest pharmacies for each SAL centroid.
sal_coords = np.column_stack([sal_proj["centroid_x"].values, sal_proj["centroid_y"].values])
distances_m, indices = tree.query(sal_coords, k = K_NEAREST)

# Store results for each k level.
for ki in range(K_NEAREST):
    col_m = f"euclidean_dist_k{ki + 1}_m"
    col_km = f"euclidean_dist_k{ki + 1}_km"
    sal_proj[col_m] = distances_m[:, ki]
    sal_proj[col_km] = distances_m[:, ki] / 1000.0

print(f"EUCLIDEAN DISTANCES COMPUTED (k = {K_NEAREST})")
print()
for ki in range(K_NEAREST):
    col_km = f"euclidean_dist_k{ki + 1}_km"
    print(f"k={ki + 1}:")
    print(sal_proj[col_km].describe().to_string())
    print()

## EUCLIDEAN DISTANCE HEATMAP

Choropleth map of SAL polygons colored by Euclidean distance
to the nearest pharmacy (k=1), shown per province.

In [ ]:
# Prepare data for mapping (keep original polygon geometry, not centroids).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

# Determine province column name from shapefile.
prov_col = None
for candidate in ["PR_NAME", "PROVINCE", "Province", "province"]:
    if candidate in sal_map_data.columns:
        prov_col = candidate
        break

if prov_col is None:
    print("WARNING: No province column found. Plotting all SALs together.")
    prov_col = "ALL"
    sal_map_data["ALL"] = "All Provinces"

print(f"Province column: {prov_col}")
print(f"Unique values: {sal_map_data[prov_col].unique()}")

In [ ]:
# Define distance bins in kilometers.
dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]

fig, axes = plt.subplots(1, 2, figsize = (18, 10))
cmap = custom_cmap

provinces_in_data = sal_map_data[prov_col].unique()

for i, prov in enumerate(provinces_in_data[:2]):
    subset = sal_map_data[sal_map_data[prov_col] == prov].copy()
    subset = subset[subset["euclidean_dist_k1_km"].notna()]

    subset.plot(
        column = "euclidean_dist_k1_km",
        cmap = cmap,
        scheme = "UserDefined",
        classification_kwds = {"bins": dist_bins_km},
        legend = True,
        legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fmt": "{:.0f}"},
        ax = axes[i],
        edgecolor = "none",
        linewidth = 0
    )

    reformat_legend_labels(axes[i])
    axes[i].set_title(f"{prov}: Euclidean Distance to Nearest Pharmacy (k = 1)")
    
    # Add scalebar.
    axes[i].add_artist(ScaleBar(dx = 1, units = "m", location = "lower left"))
    axes[i].axis("off")

plt.suptitle("SAL Centroid Euclidean Distance to Nearest Pharmacy", y = 1.02, size = 14)
plt.tight_layout()

ds.save_fig("heatmap-euclidean-k1", fig, fig_dir = IMAGES)

## LOAD SAVED NETWORK GRAPHS

Load pre-saved `.graphml` files from disk. These were downloaded
in a previous session and should not need re-downloading.

**Expected files:**
- `network_gauteng_walk.graphml`
- `network_gauteng_drive.graphml`
- `network_kwazulu_natal_walk.graphml`
- `network_kwazulu_natal_drive.graphml`

In [ ]:
# Verify all expected graphml files exist before loading.
missing = []
for key, path in GRAPHML_PATHS.items():
    exists = os.path.exists(path)
    status = "FOUND" if exists else "MISSING"
    print(f"  {status}: {path}")
    if not exists:
        missing.append(key)

if missing:
    print(f"MISSING GRAPHS: {missing}")
    print("Cannot proceed without all network graphs.")
    print("Re-run the download step or check file paths.")
else:
    print("ALL NETWORK GRAPHS FOUND")

In [ ]:
# Load all network graphs from disk.
# This is slow (large files) but much faster than re-downloading.
graphs = {}

for key, path in GRAPHML_PATHS.items():
    if not os.path.exists(path):
        print(f"SKIPPING {key}: file not found.")
        continue

    print(f"LOADING: {key}")
    start_time = time.time()
    G = ox.load_graphml(path)
    elapsed = time.time() - start_time
    print(f"  Loaded in {elapsed:.1f}s: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    graphs[key] = G

print(f"GRAPHS LOADED: {list(graphs.keys())}")

## NETWORK DISTANCE COMPUTATION (k = 3)

**Why k=3 changes the algorithm:**

k=1 used multi-source Dijkstra, seeding from all pharmacy nodes at
once. That gives the minimum distance from *any* pharmacy to each
graph node in a single pass, which is very fast, but it cannot
distinguish which pharmacy is nearest, second-nearest, or third.

k=3 requires single-source Dijkstra from **each pharmacy node
individually**, then accumulating the k smallest distances per SAL
centroid node. This is O(P x V log V) where P = pharmacy nodes
and V = reachable graph nodes within the cutoff.

**Checkpointing:** Progress is saved to disk every N pharmacies.
If the kernel crashes or is interrupted, re-running the cell will
resume from the last checkpoint rather than starting over.

**Memory management:** A max-heap of fixed size k is maintained per
SAL centroid node, so memory is bounded at O(SAL_nodes x k)
regardless of how many pharmacies are processed.

In [ ]:
def compute_network_distances_kn(G, sal_df, pharm_df, k = 3,
                                 cutoff_m = 50000, label = "network",
                                 checkpoint_dir = OUT_CHECKPOINT_DIR,
                                 checkpoint_interval = 50):
    """Compute shortest-path distances from each SAL centroid to
    the k nearest pharmacies using per-pharmacy Dijkstra with
    checkpointing.

    Parameters
    G : networkx.MultiDiGraph (unprojected, WGS84).
    sal_df : GeoDataFrame with centroid_lat and centroid_lng.
    pharm_df : GeoDataFrame with lat and lng.
    k : number of nearest pharmacies to track per SAL.
    cutoff_m : Dijkstra exploration cutoff in meters.
    label : string label for checkpoint filename.
    checkpoint_dir : directory for checkpoint pickle files.
    checkpoint_interval : save checkpoint every N pharmacies.

    Returns
    np.ndarray of shape (len(sal_df), k) with distances in meters.
    NaN where fewer than k pharmacies are reachable.
    """
    os.makedirs(checkpoint_dir, exist_ok = True)
    checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_{label}_k{k}.pkl")

    # Snap pharmacies to nearest graph nodes.
    pharm_nodes_all = ox.nearest_nodes(
        G, X = pharm_df["LNG"].values, Y = pharm_df["LAT"].values
    )
    unique_pharm_nodes = list(set(pharm_nodes_all))
    print(f"  Pharmacy nodes snapped: {len(pharm_nodes_all)} total -> {len(unique_pharm_nodes)} unique")

    # Snap SAL centroids to nearest graph nodes.
    sal_nodes = ox.nearest_nodes(
        G, X = sal_df["centroid_lng"].values, Y = sal_df["centroid_lat"].values
    )
    unique_sal_nodes = set(sal_nodes)
    print(f"  SAL centroid nodes snapped: {len(sal_nodes)} total, {len(unique_sal_nodes)} unique")

    # Convert to undirected for bidirectional traversal.
    G_undirected = G.to_undirected()

    # Load checkpoint if one exists (resume interrupted run).
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "rb") as f:
            checkpoint = pickle.load(f)
        processed = checkpoint["processed"]
        sal_heaps = checkpoint["sal_heaps"]
        print(f"  CHECKPOINT LOADED: {len(processed)}/{len(unique_pharm_nodes)} pharmacies already processed")
    else:
        processed = set()
        # Max-heap of size k per unique SAL node.
        # Store negative distances so heapq (min-heap) acts as max-heap.
        sal_heaps = {node: [] for node in unique_sal_nodes}

    remaining = [n for n in unique_pharm_nodes if n not in processed]
    total_remaining = len(remaining)
    total_pharm = len(unique_pharm_nodes)

    if total_remaining == 0:
        print(f"  ALL {total_pharm} PHARMACIES ALREADY PROCESSED (checkpoint complete)")
    else:
        print(f"  PROCESSING {total_remaining} remaining pharmacies (of {total_pharm} total)")

    # Track timing for estimates.
    batch_start = time.time()
    cumulative_dijkstra_time = 0.0

    for idx, pharm_node in enumerate(remaining):
        # Single-source Dijkstra with cutoff from this pharmacy node.
        t0 = time.time()
        dist_dict = nx.single_source_dijkstra_path_length(
            G_undirected, pharm_node, cutoff = cutoff_m, weight = "length"
        )
        cumulative_dijkstra_time += time.time() - t0

        # Update heaps for SAL nodes reached by this pharmacy.
        for sal_node in unique_sal_nodes:
            if sal_node in dist_dict:
                d = dist_dict[sal_node]
                heap = sal_heaps[sal_node]
                if len(heap) < k:
                    # Heap not full yet, push directly.
                    heapq.heappush(heap, -d)
                elif d < -heap[0]:
                    # New distance is smaller than current max in heap, replace.
                    heapq.heapreplace(heap, -d)

        processed.add(pharm_node)

        # Progress reporting every checkpoint_interval pharmacies.
        done_count = idx + 1
        if done_count % checkpoint_interval == 0 or done_count == total_remaining:
            elapsed = time.time() - batch_start
            avg_per_pharm = elapsed / done_count
            remaining_count = total_remaining - done_count
            eta_seconds = avg_per_pharm * remaining_count
            eta_minutes = eta_seconds / 60

            print(f"  PROGRESS: {len(processed)}/{total_pharm} pharmacies "
                  f"({done_count}/{total_remaining} this session) | "
                  f"avg {avg_per_pharm:.1f}s/pharm | "
                  f"ETA: {eta_minutes:.0f} min")

            # Save checkpoint.
            with open(checkpoint_path, "wb") as f:
                pickle.dump({"processed": processed, "sal_heaps": sal_heaps}, f)

    total_elapsed = time.time() - batch_start
    print(f"  DIJKSTRA COMPLETE: {total_elapsed / 60:.1f} min total, "
          f"{cumulative_dijkstra_time / 60:.1f} min in Dijkstra")

    # Extract top-k distances for each SAL centroid (in original order).
    results = np.full((len(sal_nodes), k), np.nan)
    for i, sal_node in enumerate(sal_nodes):
        # Heap contains negative distances; negate and sort ascending.
        dists_neg = sal_heaps.get(sal_node, [])
        dists = sorted([-d for d in dists_neg])
        for j in range(min(k, len(dists))):
            results[i, j] = dists[j]

    # Report reachability per k level.
    for ki in range(k):
        reachable = np.isfinite(results[:, ki]).sum()
        print(f"  k={ki + 1}: {reachable}/{len(sal_nodes)} SALs reachable "
              f"({reachable / len(sal_nodes) * 100:.1f}%)")

    # Clean up checkpoint file after successful completion.
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)
        print(f"  CHECKPOINT CLEANED UP: {checkpoint_path}")

    return results

## COMPUTE TIME ESTIMATES

Estimate total run time before starting the computation.
Rough heuristic: ~2-10 seconds per pharmacy node per Dijkstra,
depending on graph density and cutoff distance.

In [ ]:
# Estimate run time for each province x mode combination.
# Determine province column.
if prov_col is None:
    for candidate in ["PR_NAME", "PROVINCE", "Province", "province"]:
        if candidate in sal_proj.columns:
            prov_col = candidate
            break

# Rough seconds-per-pharmacy heuristic based on graph node count.
# Walk graphs are denser (more nodes) and slower than drive graphs.
def estimate_seconds_per_pharmacy(n_nodes):
    """Rough heuristic for Dijkstra time per pharmacy node."""
    if n_nodes > 1_000_000:
        return 8.0
    elif n_nodes > 500_000:
        return 5.0
    elif n_nodes > 100_000:
        return 3.0
    else:
        return 1.5

print("ESTIMATED RUN TIMES")
print(f"(Assumes {CUTOFF_M / 1000:.0f} km cutoff, k = {K_NEAREST})")
print()

total_estimate_min = 0

for prov_name in PROVINCES:
    for net_type in ["walk", "drive"]:
        graph_key = f"{prov_name}_{net_type}"
        if graph_key not in graphs:
            print(f"  {graph_key}: GRAPH NOT LOADED, skipping estimate")
            continue

        G = graphs[graph_key]
        n_nodes = G.number_of_nodes()

        # Count pharmacies for this province.
        pharm_prov_col = None
        for candidate in ["PROVINCE", "Province", "province"]:
            if candidate in pharm_gdf.columns:
                pharm_prov_col = candidate
                break

        if pharm_prov_col is not None:
            prov_mask = pharm_gdf[pharm_prov_col].str.contains(prov_name, case = False, na = False)
            n_pharm = prov_mask.sum()
        else:
            # Rough count from spatial bounds.
            prov_mask_sal = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
            prov_bounds = sal_proj[prov_mask_sal].to_crs("EPSG:4326").total_bounds
            n_pharm = len(pharm_gdf.cx[prov_bounds[0]:prov_bounds[2], prov_bounds[1]:prov_bounds[3]])

        sec_per = estimate_seconds_per_pharmacy(n_nodes)
        est_min = (n_pharm * sec_per) / 60

        # Check for existing checkpoint (already-processed pharmacies).
        cp_path = OUT_CHECKPOINT_DIR / f"checkpoint_{prov_name}_{net_type}_k{K_NEAREST}.pkl"
        already_done = 0
        if os.path.exists(cp_path):
            with open(cp_path, "rb") as f:
                cp = pickle.load(f)
            already_done = len(cp["processed"])

        remaining_pharm = max(0, n_pharm - already_done)
        adj_est_min = (remaining_pharm * sec_per) / 60

        total_estimate_min += adj_est_min

        checkpoint_note = f" ({already_done} already checkpointed)" if already_done > 0 else ""
        print(f"  {graph_key}:")
        print(f"    Graph nodes: {n_nodes:,}")
        print(f"    Pharmacies: ~{n_pharm}{checkpoint_note}")
        print(f"    Est. {sec_per:.0f}s/pharmacy -> ~{adj_est_min:.0f} min")
        print()

print(f"TOTAL ESTIMATED TIME: ~{total_estimate_min:.0f} min ({total_estimate_min / 60:.1f} hours)")
print()
print("These are rough estimates. Actual times depend on graph density,")
print("cutoff distance, and hardware. The checkpoint system ensures no")
print("progress is lost if the kernel is interrupted.")

## RUN k=3 NETWORK DISTANCE COMPUTATION

This is the long-running step. Each province x mode combination
runs independently with its own checkpoint file.

**If interrupted:** Re-run this cell. It will detect existing
checkpoints and resume from where it left off.

**After completion:** Checkpoint files are automatically deleted.

In [ ]:
# Compute k-nearest network distances for each province and network type.
for prov_name in PROVINCES:
    # Filter SALs for this province.
    prov_mask_sal = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
    sal_prov = sal_proj[prov_mask_sal].copy()

    if sal_prov.shape[0] == 0:
        print(f"WARNING: No SALs for {prov_name}. Check province column values.")
        continue

    # Assign pharmacies to provinces by location.
    prov_polys = sal_proj[[prov_col, "geometry"]].dissolve(by = prov_col).reset_index()
    pharm_located = gpd.sjoin(
        pharm_gdf.to_crs(sal_proj.crs), prov_polys, how = "inner", predicate = "within"
    ).to_crs("EPSG:4326")
    print(pharm_located[prov_col].value_counts())  # expect roughly 1,450 Gauteng and 700 KZN
    print(f"Pharmacies outside both provinces (dropped): {len(pharm_gdf) - len(pharm_located)}")

    # Compute k-nearest network distances for each province and network type.
    for prov_name in PROVINCES:
        # Filter SALs for this province.
        prov_mask_sal = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
        sal_prov = sal_proj[prov_mask_sal].copy()

        if sal_prov.shape[0] == 0:
            print(f"WARNING: No SALs for {prov_name}. Check province column values.")
            continue

        # Pharmacies located inside this province.
        pharm_prov = pharm_located[pharm_located[prov_col] == prov_name].copy()
        assert len(pharm_prov) > 100, f"Only {len(pharm_prov)} pharmacies for {prov_name}; check the join."

        print(f"PROVINCE: {prov_name}")
        print(f"  SALs: {sal_prov.shape[0]}, Pharmacies: {pharm_prov.shape[0]}")

        for net_type in ["walk", "drive"]:
            graph_key = f"{prov_name}_{net_type}"
            if graph_key not in graphs:
                print(f"  SKIPPING {graph_key}: graph not loaded.")
                continue

            G = graphs[graph_key]
            label = f"{prov_name}_{net_type}"

            print(f"  COMPUTING {net_type.upper()} NETWORK DISTANCES (k = {K_NEAREST})")
            run_start = time.time()

            results = compute_network_distances_kn(
                G = G,
                sal_df = sal_prov,
                pharm_df = pharm_prov,
                k = K_NEAREST,
                cutoff_m = CUTOFF_M,
                label = label,
                checkpoint_dir = OUT_CHECKPOINT_DIR,
                checkpoint_interval = CHECKPOINT_INTERVAL
            )

            # Store results back in the main dataframe.
            for ki in range(K_NEAREST):
                col_m = f"{net_type}_dist_k{ki + 1}_m"
                col_km = f"{net_type}_dist_k{ki + 1}_km"
                sal_proj.loc[prov_mask_sal, col_m] = results[:, ki]
                sal_proj.loc[prov_mask_sal, col_km] = results[:, ki] / 1000.0

            run_elapsed = time.time() - run_start
            print(f"  {graph_key} COMPLETE: {run_elapsed / 60:.1f} min")
            print()

            # Save intermediate CSV after each province x mode combination.
            interim_path = OUT_INTERIM_DIR / f"sal_distances_k{K_NEAREST}_interim_{label}.csv"
            interim_cols = ["EA_CODE"]
            for ki in range(K_NEAREST):
                interim_cols.extend([f"{net_type}_dist_k{ki + 1}_m", f"{net_type}_dist_k{ki + 1}_km"])
            available_interim = [c for c in interim_cols if c in sal_proj.columns]
            sal_proj.loc[prov_mask_sal, available_interim].to_csv(interim_path, index = False)
            print(f"  INTERIM SAVED: {interim_path}")
            print()

## VALIDATE k=3 RESULTS

Confirm that k=1 <= k=2 <= k=3 (monotonicity check) and report
how many SALs have fewer than k pharmacies reachable.

In [ ]:
# Monotonicity check: k=1 distance should always be <= k=2 <= k=3.
print("MONOTONICITY VALIDATION")
violations = 0

for mode in ["euclidean", "walk", "drive"]:
    for ki in range(1, K_NEAREST):
        col_prev = f"{mode}_dist_k{ki}_km"
        col_curr = f"{mode}_dist_k{ki + 1}_km"

        if col_prev not in sal_proj.columns or col_curr not in sal_proj.columns:
            continue

        # Only check where both values exist.
        mask = sal_proj[col_prev].notna() & sal_proj[col_curr].notna()
        if mask.sum() == 0:
            continue

        bad = sal_proj.loc[mask, col_curr] < sal_proj.loc[mask, col_prev]
        n_bad = bad.sum()
        violations += n_bad

        status = "PASS" if n_bad == 0 else f"FAIL ({n_bad} violations)"
        print(f"  {mode} k={ki} <= k={ki + 1}: {status}")

print(f"TOTAL VIOLATIONS: {violations}")
print()

# Reachability summary by k level.
print("REACHABILITY SUMMARY (SALs with valid distance)")
for mode in ["euclidean", "walk", "drive"]:
    for ki in range(1, K_NEAREST + 1):
        col = f"{mode}_dist_k{ki}_km"
        if col not in sal_proj.columns:
            continue
        valid = sal_proj[col].notna().sum()
        total = len(sal_proj)
        print(f"  {mode} k={ki}: {valid}/{total} ({valid / total * 100:.1f}%)")
    print()

## DISTANCE SUMMARY STATISTICS

Compare distances across k levels, provinces, and distance modes.

In [ ]:
# Summary statistics by province and k level.
for prov_name in PROVINCES:
    prov_mask = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_proj[prov_mask]

    print(f"DISTANCE STATISTICS: {prov_name.upper()} (n = {len(subset)})")
    for mode in ["euclidean", "walk", "drive"]:
        for ki in range(1, K_NEAREST + 1):
            col = f"{mode}_dist_k{ki}_km"
            if col not in subset.columns:
                continue
            stats = subset[col].describe(percentiles = [0.5, 0.75, 0.9, 0.95])
            print(f"  {mode} k={ki}: median={stats['50%']:.2f} km, "
                  f"mean={stats['mean']:.2f} km, "
                  f"p90={stats['90%']:.2f} km, "
                  f"max={stats['max']:.2f} km")
        print()
    print()

In [ ]:
# Circuity ratios (network / euclidean) for k=1.
for net_col in ["walk_dist_k1_km", "drive_dist_k1_km"]:
    euc_col = "euclidean_dist_k1_km"
    if net_col in sal_proj.columns and euc_col in sal_proj.columns:
        ratio_col = net_col.replace("_dist_k1_km", "_circuity_k1")
        mask = (sal_proj[euc_col] > 0) & sal_proj[net_col].notna()
        sal_proj.loc[mask, ratio_col] = sal_proj.loc[mask, net_col] / sal_proj.loc[mask, euc_col]
        median_ratio = sal_proj[ratio_col].median()
        print(f"MEDIAN CIRCUITY RATIO ({net_col}): {median_ratio:.2f}")

## NETWORK DISTANCE HEATMAPS

Side-by-side choropleth maps comparing k=1 distances for Euclidean,
walk, and drive methods per province.

In [ ]:
# Heatmap for each distance type (k = 2).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]
cmap = custom_cmap

# Distance columns to plot (k = 2 only).
plot_candidates = ["euclidean_dist_k2_km", "walk_dist_k2_km", "drive_dist_k2_km"]
available_cols = [c for c in plot_candidates if c in sal_map_data.columns]

for prov_name in PROVINCES:
    prov_mask = sal_map_data[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_map_data[prov_mask].copy()

    plot_cols = [c for c in available_cols if subset[c].notna().sum() > 0]
    n_plots = len(plot_cols)

    if n_plots == 0:
        print(f"No distance data available for {prov_name}.")
        continue

    fig, axes = plt.subplots(1, n_plots, figsize = (7 * n_plots, 10))
    if n_plots == 1:
        axes = [axes]

    for j, col in enumerate(plot_cols):
        label = col.replace("_dist_k2_km", "").replace("_", " ").title()

        subset.plot(
            column = col,
            missing_kwds={"color": ds.UNREACHABLE_COLOR, "label": "No Data"},
            cmap = cmap,
            scheme = "UserDefined",
            classification_kwds = {"bins": dist_bins_km},
            legend = (j == n_plots - 1),
            legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fmt": "{:.0f}"},
            ax = axes[j],
            edgecolor = "none",
            linewidth = 0
        )

        axes[j].set_title(f"{prov_name} {label} Distance (k = 2)", fontsize = 12)
        
        # Add scalebar to last subplot.
        if j == n_plots - 3:
            axes[j].add_artist(ScaleBar(dx = 1, units = "m", location = "lower left"))
        
        axes[j].axis("off")

    reformat_legend_labels(axes[j])
    plt.tight_layout(rect = [0, 0, 1, 0.92])
    #plt.suptitle(f"{prov_name}: Distance to Nearest Pharmacy by Method", fontweight = "bold", fontsize = 14, y = 0.97)

    axes[j].set_facecolor(ds.OFF_WHITE)
    ds.save_fig(f"heatmap-{prov_name.lower()}-all-methods-k2", fig, fig_dir = IMAGES)

## NETWORK GRAPH PREVIEWS

Quick visual previews of the saved network graphs.

In [ ]:
# # Plot a small preview of each saved network graph.
# for graph_key, G in graphs.items():
#     print(f"GRAPH: {graph_key}")
#     print(f"  Nodes: {G.number_of_nodes():,}")
#     print(f"  Edges: {G.number_of_edges():,}")

#     fig, ax = ox.plot_graph(
#         G,
#         figsize = (10, 10),
#         node_size = 0,
#         edge_linewidth = 0.3,
#         edge_color = "black",
#         bgcolor = "white",
#         show = False,
#         close = False
#     )

#     # Format title.
#     title_parts = graph_key.rsplit("_", 1)
#     if len(title_parts) == 2:
#         province, network = title_parts
#         title = f"{province} {network.capitalize()}"
#     else:
#         title = graph_key

#     ax.set_title("")

#     output_path = os.path.join(IMAGE_DIR, f"network_{graph_key.lower().replace('-', '_')}.png")

#     fig.patch.set_alpha(0.0)
#     ax.set_facecolor("none")
#     fig.savefig(output_path, dpi = 150, bbox_inches = "tight", facecolor = "white")
#     plt.show()
#     print(f"SAVED: {output_path}")
#     print()

## EXPORT RESULTS

Save the SAL-level distance results for downstream analysis.
Output includes k=1, k=2, k=3 distances for all three methods.

In [ ]:
# Build export column list.
export_cols = ["EA_CODE", "WardID", "sal2023_est",
               "centroid_lat", "centroid_lng"]

# Add province column.
if prov_col in sal_proj.columns:
    export_cols.insert(1, prov_col)

# Add Euclidean distance columns for all k levels.
for ki in range(1, K_NEAREST + 1):
    for suffix in ["_m", "_km"]:
        col = f"euclidean_dist_k{ki}{suffix}"
        if col in sal_proj.columns:
            export_cols.append(col)

# Add network distance columns for all k levels.
for net_type in ["walk", "drive"]:
    for ki in range(1, K_NEAREST + 1):
        for suffix in ["_m", "_km"]:
            col = f"{net_type}_dist_k{ki}{suffix}"
            if col in sal_proj.columns:
                export_cols.append(col)

# Add circuity columns.
for col in sal_proj.columns:
    if "circuity" in col:
        export_cols.append(col)

# Filter to only columns that exist.
export_cols = [c for c in export_cols if c in sal_proj.columns]

export_df = sal_proj[export_cols].copy()
export_path = OUT_DISTANCES
export_df.to_csv(export_path, index = False)

print(f"RESULTS EXPORTED: {export_path}")
print(f"Rows: {export_df.shape[0]}, Columns: {export_df.shape[1]}")
print(f"Columns: {list(export_df.columns)}")

## CHECKPOINT CLEANUP

Verify no orphaned checkpoint files remain after successful
completion. If any exist, the corresponding computation did
not finish cleanly.

In [ ]:
# Check for any remaining checkpoint files.
remaining_checkpoints = [f for f in os.listdir(OUT_CHECKPOINT_DIR) if f.endswith(".pkl")]

if remaining_checkpoints:
    print(f"WARNING: {len(remaining_checkpoints)} checkpoint file(s) found.")
    print("These indicate interrupted computations that did not complete:")
    for f in remaining_checkpoints:
        fpath = OUT_CHECKPOINT_DIR / f
        with open(fpath, "rb") as fh:
            cp = pickle.load(fh)
        print(f"  {f}: {len(cp['processed'])} pharmacies processed")
    print()
    print("Re-run the computation cell to resume from these checkpoints.")
else:
    print("NO CHECKPOINT FILES FOUND: all computations completed successfully.")